In [72]:
import math
from ast import literal_eval

import numpy as np
import plotly.graph_objs as gobj
import seaborn as sns
import torch
import torch.nn.functional as F
from gtda.diagrams import PersistenceEntropy
from gtda.homology import VietorisRipsPersistence
from gtda.plotting import plot_diagram
from matrepr import mdisplay
from plotly.subplots import make_subplots
from safetensors.torch import load_model
from scipy.linalg import expm, logm
from torch import linalg
from torch.utils.data import Subset

from nnscaling import aesthetics
from nnscaling.data import (
    DatasetConfig,
    create_data_loader,
    get_dataset_class,
)
from nnscaling.models import MLP, Autoencoder, ScaledModel
from nnscaling.utils import (
    compute_model_accuracy,
    get_device,
    safetensors_metadata_parser,
    show_matrix,
)
from nnscaling.visualization import plot_decision_boundary, plot_eigenvalues

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [73]:
model_name = "mnist_model"
scale_idx = "2"

In [74]:
model_file_path = f"/home/nsa325/work/nnscaling/model_saves/{model_name}.safetensors"
scaled_file_path = f"/home/nsa325/work/nnscaling/model_saves/scaling/loc_{scale_idx}_scaled_{model_name}.safetensors"
ae_file_path = f"/home/nsa325/work/nnscaling/model_saves/scaling/loc_{scale_idx}_autoencoder_{model_name}.safetensors"

In [75]:
# Dataset config
metadata = safetensors_metadata_parser(file_path=model_file_path)
dataset_config = DatasetConfig(
    dataset_name=metadata["dataset"],
    num_samples=5_000,
    split="train",
    seed=21,
)
DatasetClass = get_dataset_class(name=dataset_config.dataset_name)
dataset = DatasetClass(config=dataset_config)

target_class = 0
idx = torch.where(dataset.labels == target_class)[0]
subset = Subset(dataset, idx)
dataloader = create_data_loader(dataset, batch_size=5000)

In [76]:
model = MLP.load_model(model_file_path, in_features=dataset.in_features).eval()
print(f"Training Accuracy: {compute_model_accuracy(model.to('cpu'), dataset)}")

Training Accuracy: 0.9976666569709778


In [77]:
scaled_model = ScaledModel.load_model(model, scaled_file_path).eval()
print(f"Training Accuracy: {compute_model_accuracy(scaled_model.to('cpu'), dataset)}")
scaled_model.hook_model(pre=True, scaled=True, post=True)

/home/nsa325/work/nnscaling/nnscaling/models/scaler.py:142: UserWarning:

The `forward` method calls the `forward_scaled` method. For vanilla forward, call `forward_raw`.



Training Accuracy: 0.09915000200271606


In [78]:
# autoencoder = Autoencoder(nonlinearity="relu", in_features=10, observable_features=10).eval()
# _ = load_model(autoencoder, ae_file_path, device=get_device())
autoencoder = Autoencoder.load_model(ae_file_path).eval()
autoencoder

Autoencoder(
  (_encoder): Sequential(
    (0): LinearLayer(
      (linear): Linear(in_features=64, out_features=128, bias=True)
      (activation_layer): LeakyReLU(negative_slope=0.01)
    )
  )
  (_koopman_matrix): LinearLayer(
    (linear): Linear(in_features=128, out_features=128, bias=False)
  )
  (_decoder): Sequential(
    (0): LinearLayer(
      (linear): Linear(in_features=128, out_features=64, bias=True)
      (activation_layer): LeakyReLU(negative_slope=0.01)
    )
  )
)

In [79]:
def pad_act(x):
    target_size = autoencoder.encoder[0].in_features
    current_size = x.size(1)
    if current_size < target_size:
        pad_size = target_size - current_size
        x = F.pad(x, (0, pad_size), mode="constant", value=0)

    return x


def plot_multiple_diagrams(diagrams, homology_dimensions=None, plotly_params=None):
    n_diagrams = len(diagrams)
    n_cols = math.ceil(math.sqrt(n_diagrams))
    n_rows = math.ceil(n_diagrams / n_cols)

    # Extract homology dimensions
    if homology_dimensions is None:
        homology_dimensions = np.unique(
            np.concatenate([np.unique(diagram[:, 2]) for diagram in diagrams])
        )

    # Define color map for homology dimensions
    color_map = {
        dim: color for dim, color in zip(homology_dimensions, ["red", "blue", "green", "purple"])
    }

    # Global axis limits for consistency
    all_points = np.concatenate([diagram[:, :2] for diagram in diagrams if diagram.size > 0])
    min_val = np.min(all_points[~np.isinf(all_points)])
    max_val = np.max(all_points[~np.isinf(all_points)])
    padding = 0.05 * (max_val - min_val)
    axis_range = [min_val - padding, max_val + padding]

    # Create a subplot figure with controlled spacing
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=[f"Layer {i+1}" for i in range(n_diagrams)],
        horizontal_spacing=0.1,  # Reduce horizontal gap
        vertical_spacing=0.1,  # Reduce vertical gap
    )

    # Add diagrams to subplots
    for idx, diagram in enumerate(diagrams):
        row, col = divmod(idx, n_cols)
        row += 1
        col += 1

        # Filter out diagonal points
        diagram = diagram[diagram[:, 0] != diagram[:, 1]]

        # Add diagonal dashed line
        fig.add_trace(
            gobj.Scatter(
                x=axis_range,
                y=axis_range,
                mode="lines",
                line=dict(dash="dash", width=1, color="black"),
                showlegend=False,
                hoverinfo="none",
            ),
            row=row,
            col=col,
        )

        # Add homology dimension points
        for dim in homology_dimensions:
            subdiagram = diagram[diagram[:, 2] == dim]
            if subdiagram.size > 0:
                fig.add_trace(
                    gobj.Scatter(
                        x=subdiagram[:, 0],
                        y=subdiagram[:, 1],
                        mode="markers",
                        name=f"H{int(dim)}",
                        marker=dict(color=color_map[dim], size=8),
                        showlegend=(idx == 0),  # Legend only in first subplot
                        hovertemplate="Birth: %{x}<br>Death: %{y}<extra></extra>",
                    ),
                    row=row,
                    col=col,
                )

    # Update layout and axis styling
    fig.update_layout(
        height=400 * n_rows,  # Adjust dynamic height
        width=450 * n_cols,  # Adjust dynamic width
        title="Persistence Diagrams",
        plot_bgcolor="white",
        legend_title="Homology Dimensions",
    )

    fig.update_xaxes(title="Birth", range=axis_range, showgrid=True)
    fig.update_yaxes(title="Death", range=axis_range, showgrid=True)

    return fig


In [80]:
batch = next(iter(dataloader))
scaled_model.forward_raw(batch[0])
scaled_model.forward_scaled(batch[0])
acts = scaled_model.get_activations(detach=True)

In [81]:
k = scaled_model.num_scaled
padded_act = pad_act(acts[f"pre_insert_{int(scale_idx)-1}"])

observables = [autoencoder.encoder(padded_act)]
for i in range(k):
    previous = observables[i]
    current = autoencoder.koopman_matrix(previous)
    observables.append(current)

observables = torch.stack(observables).detach().numpy()

In the observable space

In [82]:
VR = VietorisRipsPersistence(homology_dimensions=[0, 1], n_jobs=8)
diagrams = VR.fit_transform(observables)
plot_multiple_diagrams(diagrams)

In the state space

In [83]:
k = scaled_model.num_scaled
padded_act = pad_act(acts[f"pre_insert_{int(scale_idx)-1}"])

predictions = autoencoder(padded_act, k=k).predictions.detach().numpy()

In [84]:
VR = VietorisRipsPersistence(homology_dimensions=[0, 1], n_jobs=8)
diagrams = VR.fit_transform(predictions)
plot_multiple_diagrams(diagrams)